# 💄 Sephora Skin-Concern Based Product Recommender

## Genel Mimari

Bu notebook, `review_text_features` ve `review_concern_level` parquet dosyalarından yola çıkarak üç katmanlı bir öneri sistemi kurar:

```
Kullanıcı Girdisi (skin_type + concern)
            │
            ▼
┌───────────────────────────┐
│  Katman 1: Aggregate      │  → weighted skor (rating × effect × confidence)
│  Scoring Baseline         │
└──────────────┬────────────┘
               │
               ▼
┌───────────────────────────┐
│  Katman 2: ML Reranking   │  → LightGBM ile feature-based reranking
│  (LightGBM)               │
└──────────────┬────────────┘
               │
               ▼
┌───────────────────────────┐
│  Katman 3: Semantic       │  → sentence-transformer embedding similarity
│  Retrieval (SBERT)        │
└──────────────┬────────────┘
               │
               ▼
       Final Ensemble Skoru
       Top-N Öneri Listesi
```

---
### Kullanılan Tablolar
- `review_text_features.parquet` → ham metin + concern tag'leri
- `review_concern_level.parquet` → concern × ürün × effect_label satır bazlı tablo

## 0. Kurulum ve Import'lar

In [ ]:
# Gerekli kütüphaneler
# !pip install lightgbm sentence-transformers scikit-learn pandas numpy matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List, Dict, Optional, Tuple

# ML
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ndcg_score

# Semantic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Görsel
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ Import başarılı')

## 1. Veri Yükleme

In [ ]:
# -------------------------------------------------------
# Yolları kendi ortamınıza göre güncelleyin
# -------------------------------------------------------
DATA_DIR = Path('../data/processed')

rtf = pd.read_parquet(DATA_DIR / 'review_text_features.parquet')
rcl = pd.read_parquet(DATA_DIR / 'review_concern_level.parquet')

print('review_text_features shape :', rtf.shape)
print('review_concern_level shape :', rcl.shape)
print()
print('review_text_features columns:')
print(rtf.dtypes)
print()
print('review_concern_level columns:')
print(rcl.dtypes)

In [ ]:
# Hızlı EDA
print('=== CONCERN DAĞILIMI ===')
print(rcl['concern'].value_counts())
print()
print('=== EFFECT LABEL DAĞILIMI ===')
print(rcl['effect_label'].value_counts())
print()
print('=== SKIN TYPE DAĞILIMI ===')
if 'skin_type' in rtf.columns:
    print(rtf['skin_type'].value_counts())
print()
print('=== UNIQUE ÜRÜN SAYISI ===')
print(f"review_text_features: {rtf['product_id'].nunique()} ürün")
print(f"review_concern_level: {rcl['product_id'].nunique()} ürün")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Concern dağılımı
concern_counts = rcl['concern'].value_counts()
axes[0].barh(concern_counts.index, concern_counts.values)
axes[0].set_title('Concern Dağılımı', fontsize=14)
axes[0].set_xlabel('Review Sayısı')

# Effect label dağılımı
effect_counts = rcl['effect_label'].value_counts()
axes[1].pie(effect_counts.values, labels=effect_counts.index, autopct='%1.1f%%')
axes[1].set_title('Effect Label Dağılımı', fontsize=14)

plt.tight_layout()
plt.show()

## 2. KATMAN 1 — Aggregate Scoring Baseline

Her `(product_id, concern, skin_type)` üçlüsü için deterministik bir skor üretiriz.

**Skor formülü:**
```
effect_weight  =  +1.0 eğer 'helped'
               =  -1.5 eğer 'worsened'   (negatif deneyim daha ağır basar)
               =  +0.3 eğer 'target_only' (concern belirtmiş ama etki yazmamış)
               =   0.0 eğer 'unknown'

raw_score = mean(rating × effect_weight × concern_confidence)
review_count_bonus = log1p(review_sayısı) / 10

aggregate_score = raw_score + review_count_bonus
```

In [ ]:
EFFECT_WEIGHTS = {
    'helped': 1.0,
    'worsened': -1.5,
    'target_only': 0.3,
    'unknown': 0.0,
}

def build_aggregate_scores(rcl: pd.DataFrame, rtf: pd.DataFrame) -> pd.DataFrame:
    """
    (product_id, concern, skin_type) bazında aggregate skor üretir.
    """
    df = rcl.copy()

    # skin_type'ı review_text_features'dan al (concern_level'da yoksa)
    if 'skin_type' not in df.columns or df['skin_type'].isna().all():
        skin_map = rtf.set_index(['author_id', 'product_id'])['skin_type'].to_dict()
        df['skin_type'] = df.apply(
            lambda r: skin_map.get((r.get('author_id'), r['product_id']), 'Unknown'),
            axis=1
        )

    df['skin_type'] = df['skin_type'].fillna('Unknown')
    df['effect_weight'] = df['effect_label'].map(EFFECT_WEIGHTS).fillna(0.0)
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(3.0)

    # Normalize rating 1-5 → 0-1
    df['rating_norm'] = (df['rating'] - 1) / 4

    # Weighted contribution per row
    df['weighted_score'] = df['rating_norm'] * df['effect_weight'] * df['concern_confidence']

    # Aggregate
    agg = df.groupby(['product_id', 'concern', 'skin_type']).agg(
        product_name=('product_name_final', 'first'),
        brand_name=('brand_name_final', 'first'),
        primary_category=('primary_category', 'first'),
        secondary_category=('secondary_category', 'first'),
        mean_weighted_score=('weighted_score', 'mean'),
        mean_rating=('rating', 'mean'),
        review_count=('product_id', 'count'),
        helped_count=('effect_label', lambda x: (x == 'helped').sum()),
        worsened_count=('effect_label', lambda x: (x == 'worsened').sum()),
        mean_confidence=('concern_confidence', 'mean'),
    ).reset_index()

    # Review sayısı bonusu (popülerlik)
    agg['review_count_bonus'] = np.log1p(agg['review_count']) / 10

    # helped_ratio ve worsened_ratio
    agg['helped_ratio'] = agg['helped_count'] / agg['review_count'].clip(lower=1)
    agg['worsened_ratio'] = agg['worsened_count'] / agg['review_count'].clip(lower=1)

    # Final aggregate score
    agg['aggregate_score'] = agg['mean_weighted_score'] + agg['review_count_bonus']

    return agg.sort_values('aggregate_score', ascending=False)


agg_scores = build_aggregate_scores(rcl, rtf)
print(f'Aggregate score tablosu: {agg_scores.shape}')
agg_scores.head(10)

In [ ]:
def recommend_baseline(concern: str, skin_type: str, top_n: int = 10) -> pd.DataFrame:
    """
    Katman 1 baseline: sadece aggregate score'a göre öneri.
    skin_type filtresi yoksa 'Unknown' ya da tüm tipler üzerinden bakar.
    """
    concern = concern.lower().strip()
    skin_type = skin_type.strip() if skin_type else None

    # Önce tam eşleşme dene
    mask = agg_scores['concern'] == concern
    if skin_type:
        skin_mask = agg_scores['skin_type'].str.lower() == skin_type.lower()
        if skin_mask.sum() > 0:
            mask = mask & skin_mask

    result = agg_scores[mask].sort_values('aggregate_score', ascending=False).head(top_n)

    return result[[
        'product_id', 'product_name', 'brand_name',
        'primary_category', 'secondary_category',
        'aggregate_score', 'mean_rating', 'review_count',
        'helped_ratio', 'worsened_ratio'
    ]]


# Test
print('=== Baseline Öneri: concern=acne, skin_type=oily ===')
recommend_baseline('acne', 'oily', top_n=5)

## 3. KATMAN 2 — LightGBM ile ML Reranking

### Yaklaşım
Aggregate score tablosunu **learning-to-rank** problemi olarak ele alıyoruz.

- **Hedef değişken:** `relevance_label` — helped_ratio ve mean_rating'den türetilmiş [0-3] arası ordinal skor
- **Query:** `concern + skin_type` kombinasyonu (her query için ürünleri sıralıyoruz)
- **Algoritma:** LightGBM `lambdarank` objective (pairwise)
- **Metrik:** NDCG@10

### Feature Engineering

In [ ]:
def build_ml_features(agg_scores: pd.DataFrame) -> pd.DataFrame:
    """
    LightGBM için feature matrix üretir.
    """
    df = agg_scores.copy()

    # ---- Relevance Label ----
    # 0 = helped_ratio < 0.1 veya worsened > 0.3  (kötü)
    # 1 = orta
    # 2 = iyi
    # 3 = çok iyi (helped_ratio yüksek, rating yüksek, yeterli review)
    conditions = [
        (df['worsened_ratio'] > 0.30) | (df['helped_ratio'] < 0.10),
        (df['helped_ratio'] >= 0.10) & (df['helped_ratio'] < 0.35),
        (df['helped_ratio'] >= 0.35) & (df['helped_ratio'] < 0.60),
        (df['helped_ratio'] >= 0.60) & (df['mean_rating'] >= 3.5),
    ]
    choices = [0, 1, 2, 3]
    df['relevance_label'] = np.select(conditions, choices, default=1)

    # ---- Sayısal feature'lar ----
    df['log_review_count'] = np.log1p(df['review_count'])
    df['rating_x_helped'] = df['mean_rating'] * df['helped_ratio']
    df['net_effect_ratio'] = df['helped_ratio'] - df['worsened_ratio']
    df['confidence_x_score'] = df['mean_confidence'] * df['mean_weighted_score']

    # Concern ve skin_type label encode
    le_concern = LabelEncoder()
    le_skin = LabelEncoder()
    df['concern_enc'] = le_concern.fit_transform(df['concern'].astype(str))
    df['skin_type_enc'] = le_skin.fit_transform(df['skin_type'].astype(str))

    # Query ID — LambdaRank için her (concern, skin_type) bir grup
    df['query_id'] = df['concern_enc'].astype(str) + '_' + df['skin_type_enc'].astype(str)
    query_le = LabelEncoder()
    df['query_id_enc'] = query_le.fit_transform(df['query_id'])

    return df, le_concern, le_skin, query_le


ml_df, le_concern, le_skin, query_le = build_ml_features(agg_scores)

FEATURE_COLS = [
    'mean_weighted_score', 'mean_rating', 'log_review_count',
    'helped_ratio', 'worsened_ratio', 'net_effect_ratio',
    'mean_confidence', 'review_count_bonus',
    'rating_x_helped', 'confidence_x_score',
    'concern_enc', 'skin_type_enc',
]

print('Feature sayısı:', len(FEATURE_COLS))
print('ML veri seti boyutu:', ml_df.shape)
print()
print('Relevance label dağılımı:')
print(ml_df['relevance_label'].value_counts().sort_index())

In [ ]:
def train_lgbm_ranker(ml_df: pd.DataFrame, feature_cols: List[str]):
    """
    LightGBM LambdaRank modeli eğitir.
    GroupKFold ile concern+skin_type gruplarına göre ayırır.
    """
    X = ml_df[feature_cols].values
    y = ml_df['relevance_label'].values
    groups = ml_df['query_id_enc'].values

    # Train/Val split — gruplar bölünmeden
    unique_groups = np.unique(groups)
    np.random.seed(42)
    val_groups = np.random.choice(unique_groups, size=int(len(unique_groups) * 0.2), replace=False)
    train_mask = ~np.isin(groups, val_groups)
    val_mask = np.isin(groups, val_groups)

    X_train, y_train, g_train = X[train_mask], y[train_mask], groups[train_mask]
    X_val, y_val, g_val = X[val_mask], y[val_mask], groups[val_mask]

    # LightGBM dataset — group bilgisi ile
    def make_group_sizes(g):
        _, counts = np.unique(g, return_counts=True)
        return counts.tolist()

    train_data = lgb.Dataset(
        X_train, label=y_train,
        group=make_group_sizes(g_train),
        feature_name=feature_cols
    )
    val_data = lgb.Dataset(
        X_val, label=y_val,
        group=make_group_sizes(g_val),
        feature_name=feature_cols
    )

    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'ndcg_eval_at': [5, 10],
        'learning_rate': 0.05,
        'num_leaves': 31,
        'min_data_in_leaf': 5,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'n_jobs': -1,
        'label_gain': [0, 1, 3, 7],  # exponential gain: 0,1,2,3 labellar için
    }

    print('LightGBM LambdaRank eğitimi başlıyor...')
    model = lgb.train(
        params,
        train_data,
        num_boost_round=500,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=True),
            lgb.log_evaluation(period=50)
        ]
    )

    print(f'\n✅ En iyi iteration: {model.best_iteration}')
    return model, X_val, y_val, g_val


lgb_model, X_val, y_val, g_val = train_lgbm_ranker(ml_df, FEATURE_COLS)

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature')
plt.title('LightGBM Feature Importance (Gain)', fontsize=14)
plt.tight_layout()
plt.show()

print(importance_df)

In [ ]:
# LightGBM skorlarını ml_df'e ekle
ml_df['lgbm_score'] = lgb_model.predict(ml_df[FEATURE_COLS].values)
print('LightGBM skoru eklendi.')
ml_df[['product_name', 'brand_name', 'concern', 'skin_type',
       'aggregate_score', 'lgbm_score', 'relevance_label']].head(10)

## 4. KATMAN 3 — Semantic Retrieval (SBERT)

Kullanıcının girdisini embed edip, her ürün için önceden hesaplanmış **concern aggregate embedding**'i ile cosine similarity hesaplıyoruz.

### Ürün embedding stratejisi
Her `(product_id, concern)` kombinasyonu için o concern'e ait tüm review'ların ortalama embedding'ini alıyoruz. Bu, ürünün söz konusu concern için "ne söylendiğini" temsil eder.

In [ ]:
SBERT_MODEL_NAME = 'all-MiniLM-L6-v2'  # Hızlı, yeterince iyi
# Daha iyi kalite için: 'all-mpnet-base-v2' veya 'multi-qa-mpnet-base-dot-v1'

print(f'SBERT modeli yükleniyor: {SBERT_MODEL_NAME}')
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)
print('✅ Model hazır')

In [ ]:
def build_product_concern_embeddings(
    rcl: pd.DataFrame,
    sbert_model: SentenceTransformer,
    text_col: str = 'normalized_text',
    batch_size: int = 128
) -> Dict:
    """
    Her (product_id, concern) çifti için ortalama SBERT embedding üretir.
    
    Dönüş: {
        (product_id, concern): np.ndarray (embedding),
        ...
    }
    """
    # Tüm unique metinleri tek seferde embed et (deduplication ile)
    all_texts = rcl[text_col].fillna('').unique().tolist()
    
    print(f'Toplam {len(all_texts)} unique metin embed ediliyor...')
    all_embeddings = sbert_model.encode(
        all_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True  # Cosine sim için normalize
    )
    text_to_emb = dict(zip(all_texts, all_embeddings))
    
    # (product_id, concern) bazında ortalama embedding hesapla
    product_concern_emb = {}
    
    for (pid, concern), group in rcl.groupby(['product_id', 'concern']):
        texts = group[text_col].fillna('').tolist()
        embs = np.array([text_to_emb.get(t, np.zeros(all_embeddings.shape[1])) for t in texts])
        avg_emb = embs.mean(axis=0)
        # Normalize
        norm = np.linalg.norm(avg_emb)
        if norm > 0:
            avg_emb = avg_emb / norm
        product_concern_emb[(pid, concern)] = avg_emb
    
    print(f'✅ {len(product_concern_emb)} adet (product_id, concern) embedding üretildi.')
    return product_concern_emb


product_concern_embeddings = build_product_concern_embeddings(rcl, sbert_model)

In [ ]:
def build_query_embedding(skin_type: str, concern: str, sbert_model: SentenceTransformer) -> np.ndarray:
    """
    Kullanıcı girdisinden query embedding üretir.
    Doğal dil cümlesi olarak formüle edilir → daha iyi retrieval.
    """
    query_text = (
        f"I have {skin_type} skin and I am looking for a product that helps with {concern}. "
        f"This product should improve {concern} and work well for {skin_type} skin type."
    )
    emb = sbert_model.encode([query_text], normalize_embeddings=True)
    return emb[0]


def get_semantic_scores(
    concern: str,
    skin_type: str,
    product_concern_embeddings: Dict,
    sbert_model: SentenceTransformer
) -> pd.Series:
    """
    Verilen concern için tüm ürünlerin semantic skorlarını döner.
    """
    query_emb = build_query_embedding(skin_type, concern, sbert_model)
    
    scores = {}
    for (pid, c), emb in product_concern_embeddings.items():
        if c == concern:
            sim = float(np.dot(query_emb, emb))  # Normalize edilmiş → cosine sim = dot product
            scores[pid] = sim
    
    return pd.Series(scores, name='semantic_score')


# Test
test_sem_scores = get_semantic_scores('acne', 'oily', product_concern_embeddings, sbert_model)
print(f'Semantic score hesaplanan ürün sayısı: {len(test_sem_scores)}')
print(test_sem_scores.sort_values(ascending=False).head())

## 5. Ensemble: Tüm Katmanları Birleştirme

Her katmanın skorunu min-max normalize edip ağırlıklı toplam ile birleştiriyoruz.

```
ensemble_score = w1 × agg_score_norm
               + w2 × lgbm_score_norm
               + w3 × semantic_score_norm

Varsayılan ağırlıklar: w1=0.3, w2=0.4, w3=0.3
```

In [ ]:
def minmax_normalize(series: pd.Series) -> pd.Series:
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mn) / (mx - mn)


def recommend_ensemble(
    concern: str,
    skin_type: str,
    ml_df: pd.DataFrame,
    product_concern_embeddings: Dict,
    sbert_model: SentenceTransformer,
    lgb_model: lgb.Booster,
    feature_cols: List[str],
    top_n: int = 10,
    w_agg: float = 0.30,
    w_lgbm: float = 0.40,
    w_sem: float = 0.30,
    min_reviews: int = 3
) -> pd.DataFrame:
    """
    Ana öneri fonksiyonu: 3 katmanı birleştirip top_n ürün döner.

    Parametreler
    ------------
    concern        : Hedef concern ('acne', 'dryness', ...)
    skin_type      : Kullanıcı cilt tipi ('oily', 'dry', 'combination', ...)
    min_reviews    : Bu concern için minimum review sayısı filtresi
    """
    concern = concern.lower().strip()
    skin_type = skin_type.strip() if skin_type else 'Unknown'

    # 1. ML tablosunu filtrele
    mask = ml_df['concern'] == concern
    # Skin type: önce tam eşleşme, yoksa tüm tipler
    skin_mask = ml_df['skin_type'].str.lower() == skin_type.lower()
    if skin_mask.sum() >= 5:
        mask = mask & skin_mask
    else:
        print(f"  ⚠️  '{skin_type}' için yeterli kayıt yok, tüm skin type'lar kullanılıyor.")

    subset = ml_df[mask & (ml_df['review_count'] >= min_reviews)].copy()

    if subset.empty:
        print(f"  ❌ '{concern}' + '{skin_type}' kombinasyonu için yeterli veri bulunamadı.")
        return pd.DataFrame()

    # 2. Semantic skorları al
    sem_scores = get_semantic_scores(concern, skin_type, product_concern_embeddings, sbert_model)
    subset['semantic_score'] = subset['product_id'].map(sem_scores).fillna(0.0)

    # 3. Normalize
    subset['agg_norm'] = minmax_normalize(subset['aggregate_score'])
    subset['lgbm_norm'] = minmax_normalize(subset['lgbm_score'])
    subset['sem_norm'] = minmax_normalize(subset['semantic_score'])

    # 4. Ensemble
    subset['ensemble_score'] = (
        w_agg * subset['agg_norm'] +
        w_lgbm * subset['lgbm_norm'] +
        w_sem * subset['sem_norm']
    )

    result = subset.sort_values('ensemble_score', ascending=False).head(top_n)

    return result[[
        'product_id', 'product_name', 'brand_name',
        'primary_category', 'secondary_category',
        'ensemble_score', 'agg_norm', 'lgbm_norm', 'sem_norm',
        'mean_rating', 'review_count', 'helped_ratio', 'worsened_ratio',
        'relevance_label'
    ]].reset_index(drop=True)


print('Ensemble öneri fonksiyonu hazır.')

In [ ]:
# ======================================================
# TEST: Birkaç farklı concern + skin_type kombinasyonu
# ======================================================

test_cases = [
    ('acne', 'oily'),
    ('dryness', 'dry'),
    ('dark_spots', 'combination'),
    ('aging', 'normal'),
    ('sensitivity', 'sensitive'),
]

for concern, skin_type in test_cases:
    print(f'\n{'='*60}')
    print(f'  🔍 Concern: {concern}  |  Skin Type: {skin_type}')
    print(f'{'='*60}')
    result = recommend_ensemble(
        concern=concern,
        skin_type=skin_type,
        ml_df=ml_df,
        product_concern_embeddings=product_concern_embeddings,
        sbert_model=sbert_model,
        lgb_model=lgb_model,
        feature_cols=FEATURE_COLS,
        top_n=5
    )
    if not result.empty:
        print(result[[
            'product_name', 'brand_name', 'ensemble_score',
            'mean_rating', 'review_count', 'helped_ratio'
        ]].to_string(index=True))

## 6. Model Değerlendirme

### Offline Metrikler
- **NDCG@5, NDCG@10** — sıralama kalitesi
- **Precision@K** — top-K içinde relevant ürün oranı
- **Coverage** — kaç farklı ürün önerilebiliyor
- **Diversity** — önerilen ürünler arasındaki semantic uzaklık

In [ ]:
def evaluate_ranking(
    ml_df: pd.DataFrame,
    product_concern_embeddings: Dict,
    sbert_model: SentenceTransformer,
    lgb_model: lgb.Booster,
    feature_cols: List[str],
    k_values: List[int] = [5, 10]
) -> pd.DataFrame:
    """
    Her concern × skin_type kombinasyonu için NDCG hesaplar.
    """
    results = []
    concerns = ml_df['concern'].unique()
    skin_types = ml_df['skin_type'].unique()

    for concern in concerns:
        for skin_type in skin_types:
            subset = ml_df[
                (ml_df['concern'] == concern) &
                (ml_df['skin_type'] == skin_type) &
                (ml_df['review_count'] >= 3)
            ].copy()

            if len(subset) < 3:
                continue

            # Semantic skorları ekle
            sem_scores = get_semantic_scores(concern, skin_type, product_concern_embeddings, sbert_model)
            subset['semantic_score'] = subset['product_id'].map(sem_scores).fillna(0.0)

            # Normalize
            subset['agg_norm'] = minmax_normalize(subset['aggregate_score'])
            subset['lgbm_norm'] = minmax_normalize(subset['lgbm_score'])
            subset['sem_norm'] = minmax_normalize(subset['semantic_score'])
            subset['ensemble_score'] = (
                0.3 * subset['agg_norm'] +
                0.4 * subset['lgbm_norm'] +
                0.3 * subset['sem_norm']
            )

            true_relevance = subset['relevance_label'].values.reshape(1, -1)

            for method, score_col in [
                ('aggregate', 'agg_norm'),
                ('lgbm', 'lgbm_norm'),
                ('semantic', 'sem_norm'),
                ('ensemble', 'ensemble_score'),
            ]:
                pred_scores = subset[score_col].values.reshape(1, -1)

                for k in k_values:
                    if len(subset) >= k:
                        ndcg = ndcg_score(true_relevance, pred_scores, k=k)
                        results.append({
                            'concern': concern,
                            'skin_type': skin_type,
                            'method': method,
                            'k': k,
                            'ndcg': ndcg,
                            'n_products': len(subset)
                        })

    return pd.DataFrame(results)


print('Değerlendirme başlıyor... (birkaç dakika sürebilir)')
eval_df = evaluate_ranking(
    ml_df, product_concern_embeddings, sbert_model, lgb_model, FEATURE_COLS
)
print(f'Değerlendirme tamamlandı: {len(eval_df)} kayıt')

In [ ]:
# Metod bazında ortalama NDCG
ndcg_summary = eval_df.groupby(['method', 'k'])['ndcg'].agg(['mean', 'std']).reset_index()
ndcg_summary.columns = ['method', 'k', 'mean_ndcg', 'std_ndcg']
ndcg_summary = ndcg_summary.sort_values(['k', 'mean_ndcg'], ascending=[True, False])
print('\n=== NDCG Özeti ===')
print(ndcg_summary.to_string(index=False))

# Görsel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, k in enumerate([5, 10]):
    subset = ndcg_summary[ndcg_summary['k'] == k].sort_values('mean_ndcg', ascending=True)
    axes[i].barh(subset['method'], subset['mean_ndcg'], xerr=subset['std_ndcg'], capsize=4)
    axes[i].set_title(f'NDCG@{k} — Metod Karşılaştırması', fontsize=12)
    axes[i].set_xlim(0, 1)
    axes[i].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='0.5 baseline')

plt.tight_layout()
plt.show()

In [ ]:
# Concern bazında NDCG@10 ensemble performance
concern_ndcg = eval_df[
    (eval_df['method'] == 'ensemble') & (eval_df['k'] == 10)
].groupby('concern')['ndcg'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
concern_ndcg.plot(kind='bar')
plt.title('Ensemble NDCG@10 — Concern Bazında', fontsize=14)
plt.ylabel('NDCG@10')
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 7. Ağırlık Optimizasyonu (Opsiyonel)

Grid search ile en iyi `(w_agg, w_lgbm, w_sem)` ağırlık kombinasyonunu bul.

In [ ]:
def optimize_weights(
    ml_df: pd.DataFrame,
    product_concern_embeddings: Dict,
    sbert_model: SentenceTransformer,
    k: int = 10
) -> Tuple[float, float, float, float]:
    """
    Birkaç ağırlık kombinasyonunu test edip en iyi NDCG@k veren üçlüyü döner.
    """
    from itertools import product as iterproduct

    # Önce tüm semantic skorları hesapla (bu pahalı, bir kez yap)
    ml_df_eval = ml_df[ml_df['review_count'] >= 3].copy()
    ml_df_eval['semantic_score'] = 0.0

    for concern in ml_df_eval['concern'].unique():
        for skin_type in ml_df_eval['skin_type'].unique():
            mask = (ml_df_eval['concern'] == concern) & (ml_df_eval['skin_type'] == skin_type)
            if mask.sum() < 3:
                continue
            sem = get_semantic_scores(concern, skin_type, product_concern_embeddings, sbert_model)
            ml_df_eval.loc[mask, 'semantic_score'] = ml_df_eval.loc[mask, 'product_id'].map(sem).fillna(0.0)

    # Normalize — group içinde
    for col, new_col in [('aggregate_score', 'agg_n'), ('lgbm_score', 'lgbm_n'), ('semantic_score', 'sem_n')]:
        ml_df_eval[new_col] = ml_df_eval.groupby(['concern', 'skin_type'])[col].transform(
            lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
        )

    weight_steps = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    best_ndcg = -1
    best_weights = (0.3, 0.4, 0.3)

    candidates = [
        (w1, w2, round(1 - w1 - w2, 1))
        for w1 in weight_steps
        for w2 in weight_steps
        if 0.0 <= round(1 - w1 - w2, 1) <= 1.0
    ]

    for w1, w2, w3 in candidates:
        ml_df_eval['ens'] = w1 * ml_df_eval['agg_n'] + w2 * ml_df_eval['lgbm_n'] + w3 * ml_df_eval['sem_n']

        ndcgs = []
        for _, grp in ml_df_eval.groupby(['concern', 'skin_type']):
            if len(grp) >= k:
                try:
                    n = ndcg_score(
                        grp['relevance_label'].values.reshape(1, -1),
                        grp['ens'].values.reshape(1, -1),
                        k=k
                    )
                    ndcgs.append(n)
                except:
                    pass

        mean_ndcg = np.mean(ndcgs) if ndcgs else 0
        if mean_ndcg > best_ndcg:
            best_ndcg = mean_ndcg
            best_weights = (w1, w2, w3)

    print(f'En iyi ağırlıklar: agg={best_weights[0]}, lgbm={best_weights[1]}, sem={best_weights[2]}')
    print(f'En iyi NDCG@{k}: {best_ndcg:.4f}')
    return best_weights[0], best_weights[1], best_weights[2], best_ndcg


# NOT: Bu hücre uzun sürebilir (semantic encoding nedeniyle)
# Çalıştırmak için aşağıdaki yorum satırını kaldırın
# w_agg_opt, w_lgbm_opt, w_sem_opt, best_ndcg = optimize_weights(ml_df, product_concern_embeddings, sbert_model)
print('(Ağırlık optimizasyonu yoruma alındı — çalıştırmak için yorum satırını kaldırın)')

## 8. Model Kaydetme

In [ ]:
import pickle
import json

MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# LightGBM modelini kaydet
lgb_model.save_model(str(MODEL_DIR / 'lgbm_ranker.txt'))
print('✅ LightGBM modeli kaydedildi.')

# Aggregate score tablosunu kaydet
ml_df.to_parquet(MODEL_DIR / 'ml_scoring_table.parquet', index=False)
print('✅ ML scoring tablosu kaydedildi.')

# Product-concern embeddings
with open(MODEL_DIR / 'product_concern_embeddings.pkl', 'wb') as f:
    pickle.dump(product_concern_embeddings, f)
print('✅ Product-concern embeddings kaydedildi.')

# Label encoder'ları kaydet
with open(MODEL_DIR / 'label_encoders.pkl', 'wb') as f:
    pickle.dump({'concern': le_concern, 'skin_type': le_skin, 'query': query_le}, f)
print('✅ Label encoders kaydedildi.')

# Konfigürasyon
config = {
    'feature_cols': FEATURE_COLS,
    'sbert_model': SBERT_MODEL_NAME,
    'default_weights': {'w_agg': 0.30, 'w_lgbm': 0.40, 'w_sem': 0.30},
    'default_min_reviews': 3,
    'effect_weights': EFFECT_WEIGHTS,
}
with open(MODEL_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('✅ Konfigürasyon kaydedildi.')

print(f'\n📁 Tüm model dosyaları: {MODEL_DIR}')

## 9. Model Yükleme ve Inference Pipeline

Production'da kullanılacak yükleme ve öneri fonksiyonları.

In [ ]:
class SephoraRecommender:
    """
    Production-ready Sephora ürün öneri motoru.

    Kullanım:
    ---------
    rec = SephoraRecommender.load(model_dir='../models')
    results = rec.recommend(concern='acne', skin_type='oily', top_n=10)
    """

    def __init__(
        self,
        lgb_model: lgb.Booster,
        ml_df: pd.DataFrame,
        product_concern_embeddings: Dict,
        sbert_model: SentenceTransformer,
        feature_cols: List[str],
        config: Dict
    ):
        self.lgb_model = lgb_model
        self.ml_df = ml_df
        self.product_concern_embeddings = product_concern_embeddings
        self.sbert_model = sbert_model
        self.feature_cols = feature_cols
        self.config = config

        self.available_concerns = sorted(ml_df['concern'].unique().tolist())
        self.available_skin_types = sorted(ml_df['skin_type'].dropna().unique().tolist())

    @classmethod
    def load(cls, model_dir: str = '../models') -> 'SephoraRecommender':
        model_dir = Path(model_dir)

        lgb_model = lgb.Booster(model_file=str(model_dir / 'lgbm_ranker.txt'))
        ml_df = pd.read_parquet(model_dir / 'ml_scoring_table.parquet')

        with open(model_dir / 'product_concern_embeddings.pkl', 'rb') as f:
            product_concern_embeddings = pickle.load(f)

        with open(model_dir / 'config.json') as f:
            config = json.load(f)

        sbert_model = SentenceTransformer(config['sbert_model'])

        return cls(
            lgb_model=lgb_model,
            ml_df=ml_df,
            product_concern_embeddings=product_concern_embeddings,
            sbert_model=sbert_model,
            feature_cols=config['feature_cols'],
            config=config
        )

    def recommend(
        self,
        concern: str,
        skin_type: str,
        top_n: int = 10,
        min_reviews: int = None,
        w_agg: float = None,
        w_lgbm: float = None,
        w_sem: float = None
    ) -> pd.DataFrame:
        """
        Ana öneri metodu.

        Parametreler
        ------------
        concern    : 'acne' | 'dryness' | 'oiliness' | 'sensitivity' |
                     'redness' | 'pores' | 'dark_spots' | 'aging' |
                     'dullness' | 'texture'
        skin_type  : 'oily' | 'dry' | 'combination' | 'normal' |
                     'sensitive' | 'Unknown'
        """
        weights = self.config.get('default_weights', {})
        w_agg = w_agg or weights.get('w_agg', 0.30)
        w_lgbm = w_lgbm or weights.get('w_lgbm', 0.40)
        w_sem = w_sem or weights.get('w_sem', 0.30)
        min_reviews = min_reviews or self.config.get('default_min_reviews', 3)

        if concern not in self.available_concerns:
            print(f"⚠️ '{concern}' bulunamadı. Mevcut: {self.available_concerns}")
            return pd.DataFrame()

        return recommend_ensemble(
            concern=concern,
            skin_type=skin_type,
            ml_df=self.ml_df,
            product_concern_embeddings=self.product_concern_embeddings,
            sbert_model=self.sbert_model,
            lgb_model=self.lgb_model,
            feature_cols=self.feature_cols,
            top_n=top_n,
            w_agg=w_agg,
            w_lgbm=w_lgbm,
            w_sem=w_sem,
            min_reviews=min_reviews
        )

    def explain(
        self,
        concern: str,
        skin_type: str,
        product_id,
        rcl: pd.DataFrame,
        n_reviews: int = 3
    ) -> Dict:
        """
        Belirli bir öneri için açıklama üretir:
        - Score breakdown
        - Bu concern için bu ürün hakkında en yüksek güvenli review'lar
        """
        row = self.ml_df[
            (self.ml_df['product_id'] == product_id) &
            (self.ml_df['concern'] == concern)
        ]
        if row.empty:
            return {}

        row = row.iloc[0]

        # İlgili review'lar
        relevant_reviews = rcl[
            (rcl['product_id'] == product_id) &
            (rcl['concern'] == concern) &
            (rcl['effect_label'] == 'helped')
        ].sort_values('concern_confidence', ascending=False).head(n_reviews)

        return {
            'product_id': product_id,
            'product_name': row.get('product_name'),
            'brand_name': row.get('brand_name'),
            'concern': concern,
            'skin_type': skin_type,
            'score_breakdown': {
                'aggregate_score': round(row.get('aggregate_score', 0), 4),
                'lgbm_score': round(row.get('lgbm_score', 0), 4),
                'mean_rating': round(row.get('mean_rating', 0), 2),
                'review_count': int(row.get('review_count', 0)),
                'helped_ratio': round(row.get('helped_ratio', 0), 3),
                'worsened_ratio': round(row.get('worsened_ratio', 0), 3),
            },
            'sample_reviews': relevant_reviews[['raw_text', 'concern_confidence', 'effect_label']].to_dict('records')
        }


# Recommender'ı oluştur
recommender = SephoraRecommender(
    lgb_model=lgb_model,
    ml_df=ml_df,
    product_concern_embeddings=product_concern_embeddings,
    sbert_model=sbert_model,
    feature_cols=FEATURE_COLS,
    config=config
)

print('✅ SephoraRecommender hazır')
print(f'Mevcut concern\'lar: {recommender.available_concerns}')
print(f'Mevcut skin type\'lar: {recommender.available_skin_types}')

In [ ]:
# Final demo
print('🌟 DEMO — Kullanıcı: oily skin, acne concern')
print('=' * 60)

recs = recommender.recommend(concern='acne', skin_type='oily', top_n=5)

if not recs.empty:
    print(recs[['product_name', 'brand_name', 'ensemble_score', 'mean_rating', 'helped_ratio', 'review_count']].to_string())
    print()

    # İlk ürünün açıklaması
    top_product_id = recs.iloc[0]['product_id']
    explanation = recommender.explain('acne', 'oily', top_product_id, rcl)
    print('\n📋 TOP-1 ÜRÜN AÇIKLAMASI:')
    print(f"  Ürün: {explanation.get('product_name')} — {explanation.get('brand_name')}")
    print(f"  Score breakdown: {explanation.get('score_breakdown')}")
    if explanation.get('sample_reviews'):
        print(f"  Örnek review: {explanation['sample_reviews'][0]['raw_text'][:200]}...")

## 10. Sonraki Adımlar ve İyileştirme Önerileri

### Kısa vadeli
- **Cold start sorunu:** Yeni ürünler için semantic retrieval'ı ön plana al (LightGBM'in az review'lı ürünlerde iyi çalışmadığı durumlar için)
- **Çoklu concern:** Kullanıcı birden fazla concern girebilmeli → per-concern skor ortalaması alınabilir
- **Skin type inference:** Kullanıcı skin type bilmiyorsa, questionnaire → rule-based skin type tahmini

### Orta vadeli
- **Personalization:** `author_id` bazlı user embedding → Matrix Factorization (ALS) veya Two-Tower modeli
- **Cross-encoder reranking:** Bi-encoder retrieval + cross-encoder (BERT) ile fine reranking
- **A/B testing altyapısı:** Farklı ağırlık kombinasyonlarını online ortamda test et

### Uzun vadeli
- **Fine-tuned SBERT:** Sephora review'larıyla domain-specific embedding modeli eğit
- **LLM-based explanation:** Her öneri için GPT/Claude ile doğal dil açıklama üret
- **Real-time reranking:** Kullanıcının tıklama geçmişine göre dinamik ağırlık güncelleme